In [6]:
import torch

# Check PyTorch setup and whether GPU is available
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

import multiprocessing
# Set multiprocessing start method (helps avoid issues with DataLoader on Mac/Linux)
multiprocessing.set_start_method("fork", force=True)

multiprocessing.set_start_method("spawn", force=True)
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Torch version: 2.11.0
CUDA available: False


In [7]:
from spuco.datasets import SpuCoMNIST
from spuco.models import model_factory
from spuco.robust_train import ERM, GroupBalanceBatchERM
from spuco.evaluate import Evaluator
from spuco.group_inference import Cluster, ClusterAlg

import torch.optim as optim
import numpy as np
import random

In [8]:
# Set seeds for reproducibility across random, numpy, and torch
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)



In [9]:
from spuco.datasets import SpuCoMNIST
from spuco.datasets.base_spuco_dataset import SpuriousFeatureDifficulty

# Create train dataset with strong spurious correlation (model can easily latch onto it)
trainset = SpuCoMNIST(
    root="data",
    split="train",
    spurious_feature_difficulty=SpuriousFeatureDifficulty.MAGNITUDE_LARGE, 
    spurious_correlation_strength=0.9,  
    classes=[[i] for i in range(10)],
    label_noise=0.0
)

# Validation dataset (used to evaluate generalization)
valset = SpuCoMNIST(
    root="data",
    split="val",
    spurious_feature_difficulty=SpuriousFeatureDifficulty.MAGNITUDE_LARGE,
    classes=[[i] for i in range(10)],
    label_noise=0.0
)
# Test dataset (final evaluation, includes worst-case groups)
testset = SpuCoMNIST(
    root="data",
    split="test",
    spurious_feature_difficulty=SpuriousFeatureDifficulty.MAGNITUDE_LARGE,
    classes=[[i] for i in range(10)],
    label_noise=0.0
)
# Actually load and prepare the datasets
trainset.initialize()
valset.initialize()
testset.initialize()

print("Train size:", len(trainset))
print("Val size:", len(valset))
print("Test size:", len(testset))

Train size: 48004
Val size: 11996
Test size: 10000


In [10]:
from spuco.models import model_factory
import torch.optim as optim

INPUT_SHAPE = (3, 28, 28)

# Initialize a LeNet model for MNIST-style images
model = model_factory(
    arch='lenet',   
    input_shape=INPUT_SHAPE,
    num_classes=10,
    pretrained=False
).to(device)

# Adam optimizer for training
optimizer = optim.Adam(model.parameters(), lr=1e-3)



In [11]:
from spuco.evaluate import Evaluator

# Evaluator for validation set (used during training)
val_evaluator = Evaluator(
    testset=valset,
    group_partition=valset.group_partition,
    group_weights=valset.group_weights,
    batch_size=256,
    model = model,
    device=device
)

#Evaluator for test set (final performance, including worst-group accuracy)
test_evaluator = Evaluator(
    testset=testset,
    group_partition=testset.group_partition,
    group_weights=testset.group_weights,
    batch_size=256,
    model = model,
    device=device
)

In [12]:
import torch.multiprocessing as mp
mp.set_start_method("spawn", force=True)


# Set sharing strategy to prevent memory/file descriptor errors
import torch.multiprocessing as mp
mp.set_sharing_strategy('file_system')

Train a model using ERM


In [13]:
from spuco.robust_train import ERM

# Step 1: Train model using standard ERM (no debiasing yet)
erm_trainer = ERM(
    model=model,
    trainset=trainset,
    batch_size=128,
    optimizer=optimizer,
    num_epochs=10,
    device=device,
    val_evaluator=None,
    verbose=True,
   
)

print("Starting training...", flush = True)
erm_trainer.train()
print("Training completed.", flush = True)

# Evaluate on validation set after ERM training
val_evaluator.evaluate()



Starting training...


Epoch 0:   0%|          | 0/376 [00:00<?, ?batch/s]/Users/rishikdurvasula/spuco-george/.venv310/lib/python3.10/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/rishikdurvasula/spuco-george/.venv310/lib/python3.10/site-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version
/Users/rishikdurvasula/spuco-george/.venv310/lib/python3.10/site-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this pack

ERM | Epoch 0 | Loss: 2.299863576889038 | Accuracy: 9.375%
ERM | Epoch 0 | Loss: 2.29655385017395 | Accuracy: 16.40625%
ERM | Epoch 0 | Loss: 2.298555850982666 | Accuracy: 21.09375%
ERM | Epoch 0 | Loss: 2.29728102684021 | Accuracy: 21.09375%
ERM | Epoch 0 | Loss: 2.2903823852539062 | Accuracy: 23.4375%
ERM | Epoch 0 | Loss: 2.280008554458618 | Accuracy: 19.53125%
ERM | Epoch 0 | Loss: 2.2705042362213135 | Accuracy: 27.34375%
ERM | Epoch 0 | Loss: 2.2736003398895264 | Accuracy: 21.875%
ERM | Epoch 0 | Loss: 2.2744321823120117 | Accuracy: 17.96875%


Epoch 0:   4%|▍         | 16/376 [00:19<03:33,  1.69batch/s, accuracy=28.90625%, loss=2.14]

ERM | Epoch 0 | Loss: 2.268990993499756 | Accuracy: 17.96875%
ERM | Epoch 0 | Loss: 2.243429660797119 | Accuracy: 22.65625%
ERM | Epoch 0 | Loss: 2.234257459640503 | Accuracy: 21.875%
ERM | Epoch 0 | Loss: 2.207130193710327 | Accuracy: 23.4375%
ERM | Epoch 0 | Loss: 2.211749792098999 | Accuracy: 14.84375%
ERM | Epoch 0 | Loss: 2.186885118484497 | Accuracy: 15.625%
ERM | Epoch 0 | Loss: 2.136904001235962 | Accuracy: 27.34375%
ERM | Epoch 0 | Loss: 2.138352870941162 | Accuracy: 28.90625%


...

ERM | Epoch 9 | Loss: 0.09061729162931442 | Accuracy: 96.09375%
ERM | Epoch 9 | Loss: 0.038259923458099365 | Accuracy: 99.21875%
ERM | Epoch 9 | Loss: 0.0480983629822731 | Accuracy: 99.21875%
ERM | Epoch 9 | Loss: 0.04130836948752403 | Accuracy: 98.4375%
ERM | Epoch 9 | Loss: 0.03801891952753067 | Accuracy: 97.65625%
ERM | Epoch 9 | Loss: 0.0055136121809482574 | Accuracy: 100.0%
ERM | Epoch 9 | Loss: 0.040797486901283264 | Accuracy: 99.21875%
ERM | Epoch 9 | Loss: 0.09470336139202118 | Accuracy: 97.65625%
ERM | Epoch 9 | Loss: 0.002963969251140952 | Accuracy: 100.0%


Epoch 9: 100%|██████████| 376/376 [00:25<00:00, 14.85batch/s, accuracy=100.0%, loss=0.00296]

Training completed.



Evaluating group-wise accuracy:   0%|          | 0/100 [00:00<?, ?it/s]/Users/rishikdurvasula/spuco-george/.venv310/lib/python3.10/site-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version
/Users/rishikdurvasula/spuco-george/.venv310/lib/python3.10/site-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version
/Users/rishikdurvasula/spuco-george/.venv310/lib/python3.10/site-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https:/

{(0, 0): 100.0,
 (0, 1): 97.47899159663865,
 (0, 2): 91.59663865546219,
 (0, 3): 97.47899159663865,
 (0, 4): 94.91525423728814,
 (0, 5): 91.52542372881356,
 (0, 6): 97.45762711864407,
 (0, 7): 96.61016949152543,
 (0, 8): 94.91525423728814,
 (0, 9): 94.91525423728814,
 (1, 0): 100.0,
 (1, 1): 100.0,
 (1, 2): 99.25925925925925,
 (1, 3): 98.51851851851852,
 (1, 4): 97.77777777777777,
 (1, 5): 97.03703703703704,
 (1, 6): 97.03703703703704,
 (1, 7): 94.81481481481481,
 (1, 8): 92.53731343283582,
 (1, 9): 99.25373134328358,
 (2, 0): 92.5,
 (2, 1): 84.03361344537815,
 (2, 2): 99.15966386554622,
 (2, 3): 98.31932773109244,
 (2, 4): 91.59663865546219,
 (2, 5): 93.27731092436974,
 (2, 6): 95.7983193277311,
 (2, 7): 92.43697478991596,
 (2, 8): 94.11764705882354,
 (2, 9): 94.11764705882354,
 (3, 0): 93.4959349593496,
 (3, 1): 86.99186991869918,
 (3, 2): 82.11382113821138,
 (3, 3): 98.3739837398374,
 (3, 4): 100.0,
 (3, 5): 91.0569105691057,
 (3, 6): 88.52459016393442,
 (3, 7): 90.1639344262295,
 (

In [14]:
# Evaluate ERM-trained model on the test set
test_evaluator.evaluate()

# Report overall performance and worst-group performance
print("ERM Average Accuracy:", test_evaluator.average_accuracy)
print("ERM Worst-Group Accuracy:", test_evaluator.worst_group_accuracy)

Evaluating group-wise accuracy:   0%|          | 0/100 [00:00<?, ?it/s]/Users/rishikdurvasula/spuco-george/.venv310/lib/python3.10/site-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version
/Users/rishikdurvasula/spuco-george/.venv310/lib/python3.10/site-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version
/Users/rishikdurvasula/spuco-george/.venv310/lib/python3.10/site-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://

ERM Average Accuracy: 91.76999999999995
ERM Worst-Group Accuracy: ((9, 7), 54.45544554455446)


In [15]:
import torch
from torch.utils.data import DataLoader

# Switch model to evaluation mode (no dropout, batchnorm updates, etc.)
model.eval()

loader = DataLoader(trainset, batch_size=256, shuffle=False)

all_outputs = []
all_labels = []

# Collect model outputs for entire training set (used later for clustering)
with torch.no_grad():
    for x, y in loader:
        x = x.to(device)
        out = model(x)              
        all_outputs.append(out.cpu())
        all_labels.append(y)

# Combine all batches into single tensors
train_outputs = torch.cat(all_outputs)
train_labels = torch.cat(all_labels)

print(train_outputs.shape)
print(train_labels.shape)

torch.Size([48004, 10])
torch.Size([48004])


Cluster inputs based on the output they produce for ERM


In [16]:
from spuco.group_inference import Cluster, ClusterAlg

# Use model outputs (logits) as features for clustering
Z = train_outputs.detach().cpu().float()
labels = train_labels.detach().cpu().view(-1).tolist()

#Cluster examples within each class into 2 groups
cluster = Cluster(
    Z,
    class_labels=labels,
    num_clusters=2,
    cluster_alg=ClusterAlg.KMEANS,
    device=torch.device("cpu"),
    verbose=True
)
# Infer group assignments for each training example
group_partition_hat = cluster.infer_groups()

Clustering class-wise: 100%|██████████| 10/10 [00:00<00:00, 41.68it/s]


In [17]:
print(len(group_partition_hat))

20


Retrain using "Group-Balancing" to ensure in each batch each group appears equally

In [18]:
from spuco.datasets import GroupLabeledDatasetWrapper
# Wrap original dataset with inferred group labels from clustering
trainset_gb = GroupLabeledDatasetWrapper(
    dataset=trainset,
    group_partition=group_partition_hat
)

In [19]:

import spuco.utils.trainer as trainer_module
from torch.utils.data import DataLoader

# Override DataLoader to force single-process loading (avoids multiprocessing issues)
class PatchedDataLoader(DataLoader):
    def __init__(self, *args, **kwargs):
        kwargs["num_workers"] = 0
        super().__init__(*args, **kwargs)

# Replace default DataLoader used inside SpuCo trainer
trainer_module.DataLoader = PatchedDataLoader

In [24]:
# Retrain using group-balanced batches (each group equally represented)

gb_trainer = GroupBalanceBatchERM(
    model=model,
    trainset=trainset,
    group_partition=group_partition_hat,
    batch_size=128,
    optimizer=optimizer,
    num_epochs=10,
    device=device,
    verbose=True
)
# Train model with group balancing to reduce reliance on spurious features
gb_trainer.train()

Epoch 0:   0%|          | 0/376 [00:00<?, ?batch/s]/Users/rishikdurvasula/spuco-george/.venv310/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Epoch 0:   2%|▏         | 7/376 [00:00<00:11, 33.41batch/s, accuracy=98.4375%, loss=0.0305] 

GB | Epoch 0 | Loss: 0.044653818011283875 | Accuracy: 98.4375%
GB | Epoch 0 | Loss: 0.04077520966529846 | Accuracy: 98.4375%
GB | Epoch 0 | Loss: 0.03788981959223747 | Accuracy: 99.21875%
GB | Epoch 0 | Loss: 0.030915986746549606 | Accuracy: 99.21875%
GB | Epoch 0 | Loss: 0.022610502317547798 | Accuracy: 99.21875%
GB | Epoch 0 | Loss: 0.009382200427353382 | Accuracy: 100.0%
GB | Epoch 0 | Loss: 0.031104248017072678 | Accuracy: 98.4375%
GB | Epoch 0 | Loss: 0.01151676569133997 | Accuracy: 100.0%
GB | Epoch 0 | Loss: 0.03047793172299862 | Accuracy: 98.4375%


Epoch 0:   5%|▍         | 17/376 [00:00<00:09, 39.57batch/s, accuracy=100.0%, loss=0.0115]   

GB | Epoch 0 | Loss: 0.02731233835220337 | Accuracy: 100.0%
GB | Epoch 0 | Loss: 0.012547852471470833 | Accuracy: 100.0%
GB | Epoch 0 | Loss: 0.037589773535728455 | Accuracy: 98.4375%
GB | Epoch 0 | Loss: 0.00839396845549345 | Accuracy: 100.0%
GB | Epoch 0 | Loss: 0.012129677459597588 | Accuracy: 99.21875%
GB | Epoch 0 | Loss: 0.028505437076091766 | Accuracy: 99.21875%
GB | Epoch 0 | Loss: 0.004071473143994808 | Accuracy: 100.0%
GB | Epoch 0 | Loss: 0.03044859692454338 | Accuracy: 98.4375%
GB | Epoch 0 | Loss: 0.011519143357872963 | Accuracy: 100.0%


Epoch 0:   7%|▋         | 27/376 [00:00<00:08, 42.62batch/s, accuracy=98.4375%, loss=0.0401] 

...

GB | Epoch 9 | Loss: 0.009351393207907677 | Accuracy: 99.21875%


Epoch 9:  98%|█████████▊| 370/376 [00:08<00:00, 44.25batch/s, accuracy=100.0%, loss=0.00257]   

GB | Epoch 9 | Loss: 0.0005344254896044731 | Accuracy: 100.0%
GB | Epoch 9 | Loss: 0.0012830880004912615 | Accuracy: 100.0%
GB | Epoch 9 | Loss: 0.0027065975591540337 | Accuracy: 100.0%
GB | Epoch 9 | Loss: 0.006211130879819393 | Accuracy: 100.0%
GB | Epoch 9 | Loss: 0.002573339967057109 | Accuracy: 100.0%


Epoch 9: 100%|██████████| 376/376 [00:08<00:00, 46.29batch/s, accuracy=100.0%, loss=2.72e-5]

GB | Epoch 9 | Loss: 0.002362790983170271 | Accuracy: 100.0%
GB | Epoch 9 | Loss: 2.7178295567864552e-05 | Accuracy: 100.0%


In [26]:

from torch.utils.data import DataLoader
# Reset test loaders to use single-worker loading (avoids multiprocessing issues)
for key in test_evaluator.testloaders:
    dataset = test_evaluator.testloaders[key].dataset
    test_evaluator.testloaders[key] = DataLoader(
        dataset,
        batch_size=256,
        shuffle=False,
        num_workers=0
    )

# run evaluation
test_evaluator.model = model
test_evaluator.evaluate()

print("GB Average Accuracy:", test_evaluator.average_accuracy)
print("GB Worst-Group Accuracy:", test_evaluator.worst_group_accuracy)

Evaluating group-wise accuracy: 100%|██████████| 100/100 [00:00<00:00, 231.87it/s]

GB Average Accuracy: 95.35
GB Worst-Group Accuracy: ((3, 5), 78.21782178217822)
